# Automated EDA Report Generator
Run all cells, set `dataset_path` in the last cell.

In [ ]:

import os, json, traceback
from pathlib import Path
from datetime import datetime
import pandas as pd

try:
    from ydata_profiling import ProfileReport
except: ProfileReport=None
try:
    import sweetviz as sv
except: sv=None
try:
    from autoviz.AutoViz_Class import AutoViz_Class
except: AutoViz_Class=None
try:
    import dtale
except: dtale=None

def load_dataset(path):
    p=Path(path)
    ext=p.suffix.lower()
    if ext in [".csv",".txt",".tsv"]:
        for enc in ["utf-8","utf-8-sig","latin1","cp1252"]:
            try:
                return pd.read_csv(p,sep=None,engine="python",encoding=enc)
            except: pass
        raise RuntimeError("Cannot read file")
    if ext in [".xls",".xlsx"]: return pd.read_excel(p)
    if ext==".json": return pd.read_json(p)
    if ext==".parquet": return pd.read_parquet(p)
    if ext==".feather": return pd.read_feather(p)
    if ext==".pkl": return pd.read_pickle(p)
    raise ValueError(ext)

def generate_reports(dataset_path):
    ds=Path(dataset_path)
    out=Path("reports")/ds.stem
    out.mkdir(parents=True,exist_ok=True)
    log=[]
    def w(msg):
        print(msg); log.append(msg)
    df=load_dataset(dataset_path)
    w(f"Shape: {df.shape}")
    df.dtypes.rename("dtype").to_csv(out/"dataset_info.csv")
    df.describe(include="all").T.to_csv(out/"dataset_summary.csv")
    df.isna().sum().rename("missing").to_csv(out/"missing_values.csv")
    num=df.select_dtypes("number")
    if not num.empty:
        num.corr(numeric_only=True).to_csv(out/"correlation_matrix.csv")
    with open(out/"dataset_summary.txt","w") as f:
        f.write(str(df.info(buf=f)))
    files=[]
    if ProfileReport:
        try:
            p=ProfileReport(df,explorative=True,minimal=len(df)>100000)
            fp=out/f"{ds.stem}_ydata_profiling.html"
            p.to_file(fp)
            files.append(str(fp))
        except: traceback.print_exc()
    if sv:
        try:
            rep=sv.analyze(df)
            fp=out/f"{ds.stem}_sweetviz.html"
            rep.show_html(str(fp),open_browser=False)
            files.append(str(fp))
        except: traceback.print_exc()
    if AutoViz_Class:
        try:
            ad=out/"autoviz"; ad.mkdir(exist_ok=True)
            av=AutoViz_Class()
            os.chdir(ad)
            av.AutoViz(filename="",dfte=df,depVar="",verbose=1)
            os.chdir(Path.cwd().parent)
        except: traceback.print_exc()
    if dtale:
        try:
            inst=dtale.show(df,subprocess=False)
            w(f"D-Tale: {inst.main_url() if hasattr(inst,'main_url') else inst._main_url}")
        except: traceback.print_exc()
    manifest={"dataset":ds.name,"shape":df.shape,"generated":datetime.now().isoformat(),"files":files}
    json.dump(manifest,open(out/"manifest.json","w"),indent=2)
    open(out/"log.txt","w").write("\n".join(log))
    return out


In [ ]:
dataset_path="path/to/dataset.csv"
out=generate_reports(dataset_path)
print(out)